# Teaching a hexapod what it cannot see

A camera is a confident liar. It tells you about surfaces — *there is something 2.1 metres that way* — and says
nothing at all about what sits behind them. For a six-legged robot picking its way across a room, the difference
between **empty** and **I have no idea** is the difference between a step and a fall.

This is the VVAE: a short burst of posed camera frames in, a 3D belief about the space around the robot out.
Only the encoder ever ships to the Pi. The decoder exists to train it and then gets thrown away.

Run top to bottom. The last two sections need nothing but torch.


## Setup


In [ ]:
%load_ext autoreload
%autoreload 3
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from hexai.data import TartanGroundWindowDataset, find_trajectory_dirs
from hexai.utilities import carve_window, lift_window

ROOT = Path('../../../../../data/tartanground/House/Data_omni')
BOUNDS = torch.tensor([[-2.0, -1.5, 0.0], [2.0, 1.5, 4.0]])  # metres, in the middle frame's camera
GRID = (32, 24, 32)  # 12.5 cm cubic voxels at that box

data = TartanGroundWindowDataset(find_trajectory_dirs(ROOT), num_frames=7, image_hw=(64, 64),
                                 include_pose=True)  # poses come back relative to the middle frame
frames, poses = data[120]
print(f'{frames.shape[1]} frames, {poses[-1, :3].sub(poses[0, :3]).norm():.2f} m of travel end to end')


## What one window holds

Seven frames, about 0.21 m apart. That spacing is not a knob — it is what buys parallax. At 2 m a 5 cm baseline
moves a pixel 4.4 px, and the SG90 servos in the head miss by 3°, which is 9.9 px of pointing error. Anything
closer together and the noise swamps the signal. The head turning does not help at all: rotation gives exactly
zero parallax, so only the body walking buys depth.


In [ ]:
fig, axes = plt.subplots(2, frames.shape[1], figsize=(15, 4.6))
for t in range(frames.shape[1]):
    axes[0, t].imshow(frames[:3, t].permute(1, 2, 0).numpy())
    axes[0, t].set_title(f'{poses[t, :3].sub(poses[0, :3]).norm():.2f} m', fontsize=9)
    axes[1, t].imshow(frames[3, t].numpy(), cmap='viridis', vmin=0, vmax=1)
for ax in axes.flat: ax.set(xticks=[], yticks=[])
fig.tight_layout()


## Lifting: depth places the point, it is not carried along

Every pixel becomes a ray, its depth places it in space, and its pose drops it into one shared box that travels
with the robot. The box is anchored on the window's **middle** frame, not the first — the robot turns as it walks,
so a first-frame anchor lets the box hang off one end of the window. Middle-anchoring keeps 86.8% of points inside
against 84.6%, and cuts the query camera's spread from 0.214 m to 0.085 m.

The thing I keep having to re-explain to myself: depth is spent on *position*, never stored as a value. A voxel's
index already is its depth. Scattering the number too would bake distance-from-whichever-camera-happened-to-see-it
into a representation whose whole job is to not care about viewpoint.


In [ ]:
volume = lift_window(frames, poses, BOUNDS, GRID, 90.0)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(frames[:3, 3].permute(1, 2, 0).numpy())
axes[0].set_title('what the middle camera saw')
axes[1].imshow(volume[-1].sum(1).T.numpy(), cmap='magma', origin='lower')
axes[1].set_title('the box it built, from above')
for ax in axes: ax.set(xticks=[], yticks=[])
fig.tight_layout()


## Carving: the third state is the whole point

A lift alone gives you two states — there is stuff here, or there is no stuff here. That second one is a lie, and
it is the lie that matters. Most of the box is not empty, it is *occluded*.

So we march from the camera to each measured surface and split the box three ways: **free** where a ray passed
through, **occupied** where it stopped, **unknown** behind the surface or outside the view. How much lands in each
depends entirely on the room — a corridor and an open hall are not close — so the cell below prints the split for
this window rather than me quoting an average at you.

Keeping those apart is what lets the model decline to guess.


In [ ]:
carved = carve_window(frames, poses, BOUNDS, GRID, 90.0)
occupied, observed = carved[0], carved[1]
state = torch.zeros_like(occupied)          # 0 unknown
state[observed > 0.5] = 1.0                 # 1 free
state[occupied > 0.5] = 2.0                 # 2 occupied

heights = [4, 9, 14, 19]
fig, axes = plt.subplots(1, len(heights), figsize=(14, 3.6))
for ax, y in zip(axes, heights):
    ax.imshow(state[:, y].T.numpy(), cmap='viridis', origin='lower', vmin=0, vmax=2)
    ax.set_title(f'{BOUNDS[0, 1] + y * 0.125:.2f} m up', fontsize=9)
    ax.set(xticks=[], yticks=[])
fig.suptitle('dark unknown, teal free, yellow occupied — camera at bottom centre', fontsize=10)
fig.tight_layout()
print(f'unknown {(state == 0).float().mean():.1%}  free {(state == 1).float().mean():.1%}  occupied {(state == 2).float().mean():.1%}')


## The grid is bounded by point density, not by what looks precise

This one bit me. It is tempting to crank the resolution — finer voxels, sharper geometry, obviously better. Except
finer voxels do not subdivide a surface, they *perforate* it. You only have so many points, and spreading them over
more voxels eventually means most voxels get none.

Below about 16 points per occupied voxel, surfaces start coming apart. Which is how I found out the config I had
been running for weeks was under-sampled: I raised the vertical resolution to read clearance under furniture and
never raised the image size to match.


In [ ]:
# resolution and image size have to move together, and mine had come apart
OCC = 0.04  # occupied fraction of the box, the basis the constraint was originally measured on

def per_voxel(n_frames, px, grid):
    return n_frames * px * px / (grid[0] * grid[1] * grid[2] * OCC)

configs = [('the reference measurement', 5, 160, (32, 24, 32)),
           ('too fine for its points',   5, 160, (96, 72, 96)),
           ('what I had been running',   7,  64, (32, 60, 32)),
           ('same grid, bigger frames',  7, 128, (32, 60, 32))]

for label, n, px, grid in configs:
    d = per_voxel(n, px, grid)
    verdict = 'perforates' if d < 16 else 'ok'
    print(f'{label:26} {n} x {px:3}px into {str(grid):12} -> {d:6.1f} pts/voxel  {verdict}')


## The loss was asking for the blur

For a long time the decoder rendered soft, smeary depth and I read that as *not trained enough* or *not enough
capacity*. It was neither. It was MSE doing exactly what MSE does.

Squared error is minimised by the **conditional mean**. So when a ray is genuinely ambiguous — a wall at 1 m in
some scenes, open to 3 m in others — the cheapest answer is 2 m. A surface that was never there in either world,
but cheap in expectation. A misplaced edge costs the square of its full contrast; blurring it costs the square of
half.

L1 is minimised by the **median**, which for the same ambiguous ray is one of the two real surfaces. It commits.
Here is the whole argument in one plot, no training required.


In [ ]:
# what depth does each loss actually prefer, when the same ray is ambiguous?
from hexai.models.vvae.train import RESIDUALS

near, far = 0.25, 0.75  # 1 m and 3 m, normalised over a 4 m box
targets = torch.tensor([near] * 8 + [far] * 8)
guesses = torch.linspace(0.05, 1.0, 400)

fig, ax = plt.subplots(figsize=(7, 4))
for name in ('mse', 'l1'):
    cost = torch.stack([RESIDUALS[name](g - targets).sum() for g in guesses])
    pick = float(guesses[int(cost.argmin())])
    ax.plot(guesses * 4, cost / cost.max(), label=f'{name} -> commits to {pick * 4:.2f} m')
    ax.axvline(pick * 4, ls=':', lw=1, color=ax.lines[-1].get_color())
for d in (near, far): ax.axvline(d * 4, color='k', lw=1, alpha=0.3)
ax.set(xlabel='depth the model renders (m)', ylabel='loss, normalised',
       title='black lines are the two surfaces that actually occur')
ax.legend()
fig.tight_layout()


## What it actually produces today

Everything above is exact geometry — no network involved, and it works because it is arithmetic. This part is the
learned half, loaded from the most recent full run: 200 epochs on onecycle over eleven buildings, holding out
Office entirely, with the head jittering and monocular-grade depth noise on the input.

I load the **last** epoch rather than the one labelled best. That run's `-best.pt` is epoch 11, and epoch 11 only
won because a single validation pass swings about as far as the whole run improves. Picking the best raw epoch
picks the luckiest draw. The last epoch reads +0.526 answerable skill and keeps reading it.


In [ ]:
import json

from hexai.data import split_context_target
from hexai.models.vvae import VVAEGeometric, VVAEGeometricConfig

RUNS = Path('../.runs')
CKPT = RUNS / 'geo-c32g4-depth-onecycle-b0.0001-lr0.003-jit-unk-lift-g32x60x32c4.pt'
state = torch.load(CKPT, weights_only=False)
print(f"epoch {state['epoch']}, trained on {len(state['config']['environments'])} buildings, "
      f"holding out {state['config']['val_env']}")

# the model has to be rebuilt exactly as it was trained: 5 cm vertical voxels, the robot's 49 deg lens
val = TartanGroundWindowDataset(find_trajectory_dirs(Path('../../../../../data/tartanground/Office/Data_omni')),
                                num_frames=7, image_hw=(64, 64), include_pose=True,
                                head_fovy_deg=49.0, include_head_jitter=True)
cfg = VVAEGeometricConfig(render_hw=(64, 64), n_downsamples=2, downsample_scale=2,
                          grid_resolution=(32, 60, 32), latent_channels=4, kernel_size=3,
                          base_channels=32, groups=4, target='depth', predict_unknown=True,
                          learned_lift=True, source_fovy_deg=49.0)
model = VVAEGeometric(cfg)
model.load_state_dict(state['model'])
model.eval()
print(f'{sum(p.numel() for p in model.encoder.parameters()):,} encoder params — the only half that ships')


### The held-out view

The decoder is handed a pose it was never shown and asked to render the depth from there. That is the whole test:
reconstructing the frames you were given is satisfied by a latent that just stores pictures, and only a held-out
viewpoint forces it to store a *scene*.

This is also where I have to be straight with you about how far along it is.


In [ ]:
frames_v, poses_v = val[40]
context, context_poses, truth, query = split_context_target(frames_v[None], poses_v[None], 1, 'depth')
with torch.no_grad():
    out = model(context, context_poses, query)

render, target = out.x_hat[0, -1], truth[0, -1]
err = (render - target).abs()
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax, img, title in zip(axes, (target, render, err),
                          ('truth, a pose it never saw', 'what it renders', 'absolute error')):
    ax.imshow(img.numpy(), cmap='viridis' if title != 'absolute error' else 'inferno', vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.set(xticks=[], yticks=[])
fig.tight_layout()
print(f'mean depth error {err.mean() * 4:.2f} m over a 4 m box')


## Where it actually stands

Honestly? Not there yet. The geometry is verified and the machinery works — the carving separates seen from unseen
with a +0.70 gap, the decoded volume matches what was observed, and a schedule fix took answerable skill from
+0.395 to +0.526. But the render still tracks the query pose without holding much structure, and depth error
averages about 1.0 m across the whole held-out building — the window above is a kinder one than most. Beating a
mean-image baseline is not the same as rendering a room.

Next up, both in this repo's `.agents/todo.md`: fix the point density above, then find out whether L1 actually buys
the commitment the plot says it should. There is a real chance it does not and the answer is somewhere else
entirely — that is roughly how the last four ideas went.

The occlusion penalty, for instance, was going to be the fix. It worked perfectly and made everything worse: cut
hidden density by 94% and hollowed the room out to a sixth of the geometry that went in. Some levers are just
levers.
